In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder\
                .appName("Tiki_Crawler")\
                .getOrCreate()

print("Spark version:", spark.version)

Spark version: 3.5.0


In [3]:
import requests

# 1. Mục tiêu: API Chi tiết 1 sản phẩm (Product Detail)
url = "https://api.tiki.vn/raiden/v3/widgets/product/recentlyviewed?product_id=74419727&ids=74419727,278932606,278016820,278439196&platform=desktop"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

print("Đang chui vào kho Tiki lấy hồ sơ chi tiết sản phẩm...")
response = requests.get(url, headers=headers)

if response.status_code == 200:
    # Cục JSON trả về bây giờ chính là 1 cái Object Sản Phẩm
    product_detail = response.json().get('data', [])

    for pro in product_detail:
        
        print("Bắt thành công!")
        print("-" * 40)
        
        # Bóc tách trực tiếp từ Object
        print("Tên SP:", pro.get('name'))
        print("Giá gốc:", pro.get('original_price'), "VND")
        print("Giá đang bán:", pro.get('price'), "VND")
        
        # Bóc thử xem nó đang giảm giá bao nhiêu %
        discount_rate = pro.get('discount_rate', 0)
        print(f"Khuyến mãi: Giảm {discount_rate}%")
        
        # Lấy thử 150 ký tự đầu tiên của đoạn Mô tả sản phẩm
        short_desc = pro.get('short_description', '')
        print("Mô tả ngắn:", short_desc[:360], "...")
        
        print("-" * 40)
else:
    print("Lỗi rồi! Mã trạng thái:", response.status_code)

Đang chui vào kho Tiki lấy hồ sơ chi tiết sản phẩm...
Bắt thành công!
----------------------------------------
Tên SP: Combo 5 Quần Lót Nữ Ren Hoa Elegant Miley Lingerie FLS-03
Giá gốc: 410000 VND
Giá đang bán: 375000 VND
Khuyến mãi: Giảm 9%
Mô tả ngắn:  ...
----------------------------------------
Bắt thành công!
----------------------------------------
Tên SP:  Quần Legging Nữ 2 Trong 1 Có Túi Co Giãn Thắt Lưng Tập Luyện Quần Tập Yoga Tập Thể Dục Tập Gym Thể Thao
Giá gốc: 255000 VND
Giá đang bán: 239000 VND
Khuyến mãi: Giảm 6%
Mô tả ngắn:  ...
----------------------------------------
Bắt thành công!
----------------------------------------
Tên SP: ÁO LÓT BẦU COTTON SIÊU ĐẸP SIZE 38-42 DÀNH CHO NỮ
Giá gốc: 70000 VND
Giá đang bán: 70000 VND
Khuyến mãi: Giảm 0%
Mô tả ngắn:  ...
----------------------------------------
Bắt thành công!
----------------------------------------
Tên SP: Màn Hình KTC H27T13 2K 27 inch QHD 120Hz NEW 7ms - Hàng Chính Hãng
Giá gốc: 3450000 VND
Giá đang bán: 2590

In [4]:
import requests
from pyspark.sql import SparkSession
from pyspark.sql.types import Row

# 1. Khởi động lại động cơ Spark (Phòng khi bạn đã tắt máy)
spark = SparkSession.builder.appName("Tiki_Data_Parser").getOrCreate()

# 2. Bắn API lấy dữ liệu thô
url = "https://api.tiki.vn/falcon/ext/v1/products/comparisons?mpid=278439196&spid=278439197&cate_id=1846"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

print("Đang hút dữ liệu từ Tiki...")
response = requests.get(url, headers=headers)
raw_json = response.json()

# Móc vào key 'products' bên trong key 'data'
products_list = raw_json.get("data", {}).get("products", [])

# 3. Data Transformation (Chế biến dữ liệu từ lộn xộn thành phẳng)
cleaned_data = []

for p in products_list:
    # Lấy các thông tin lớp ngoài
    name = p.get("name")
    price = p.get("price")
    seller = p.get("current_seller", {}).get("name", "N/A")
    
    # Tạo biến tạm để hứng dữ liệu từ mảng attributes
    brand = "N/A"
    resolution = "N/A"
    refresh_rate = "N/A"
    
    # Vòng lặp "đập phẳng" (flatten) mảng attributes
    for attr in p.get("attributes", []):
        code = attr.get("code")
        value = attr.get("value")
        
        if code == "brand":
            brand = value
        elif code == "resolution":
            resolution = value
        elif code == "thoi_gian_dap_ung" or code == "toc_do_lam_tuoi":
            refresh_rate = value

    # Gói tất cả vào một DTO (Data Transfer Object) dạng Row của Spark
    row_data = Row(
        Ten_SP=name, 
        Gia_VND=price, 
        Nha_Ban=seller, 
        Thuong_Hieu=brand, 
        Do_Phan_Giai=resolution, 
        Tan_So_Quet=refresh_rate
    )
    cleaned_data.append(row_data)

# 4. Phép màu của Spark: Biến List Python thành Spark DataFrame
df = spark.createDataFrame(cleaned_data)

# 5. Hiển thị thành quả lên màn hình
print("✅ Biến đổi thành công! Đây là cấu trúc bảng hoàn chỉnh:")
# tham số truncate=False giúp in ra đầy đủ chữ không bị cắt ngắn thành dấu ...
df.show(truncate=False)

Đang hút dữ liệu từ Tiki...
✅ Biến đổi thành công! Đây là cấu trúc bảng hoàn chỉnh:
+-----------------------------------------------------------------------+-------+------------+-----------+--------------------+---------------------+
|Ten_SP                                                                 |Gia_VND|Nha_Ban     |Thuong_Hieu|Do_Phan_Giai        |Tan_So_Quet          |
+-----------------------------------------------------------------------+-------+------------+-----------+--------------------+---------------------+
|Màn Hình KTC H27T13 2K 27 inch QHD 120Hz NEW 7ms - Hàng Chính Hãng     |2590000|Tiki Trading|KTC        |WQHD IPS 2560 x 1440|1ms (MPRT), 7ms (GTG)|
|Màn hình Xiaomi A27UI 27 inch (4K/IPS/60HZ/6MS) - Hàng Chính Hãng      |5890000|Tiki Trading|Xiaomi     |UHD 4K (3840 x 2160)|1ms                  |
|Màn Hình Viewsonic VA2732A-H 120Hz 27inch Full HD IPS - Hàng Chính Hãng|2350000|Tiki Trading|Viewsonic  |1920 x 1080 pixels  |1ms                  |
|Màn Hình Viewso

In [20]:
import requests
url = "https://api.tiki.vn/raiden/v3/widgets/product/recentlyviewed?product_id=278439196&ids=278439196&platform=desktop"

headers ={
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}
print("Chi tiết sản phẩm")

response = requests.get(url, headers=headers)

if response.status_code == 200:
    product_detail = response.json().get('data', [])

    product_detail = product_detail[0]
    print("Tên sản phẩm:", product_detail.get('name', ''))
    print("Giá sản phẩm:", product_detail.get('price', ''))
else:
    print("Lỗi dồi")

Chi tiết sản phẩm
Tên sản phẩm: Màn Hình KTC H27T13 2K 27 inch QHD 120Hz NEW 7ms - Hàng Chính Hãng
Giá sản phẩm: 2590000


NameError: name 'spark' is not defined